In [20]:
!pip install stanza==1.2.1
!pip install spicy hazm==0.9.2

  Attempting uninstall: stanza
    Found existing installation: stanza 1.10.1
    Uninstalling stanza-1.10.1:
      Successfully uninstalled stanza-1.10.1


In [36]:
# preprocessing/pos_tagger.py

import nltk
import stanza

class POSTagger:
    def __init__(self, lang: str = "en"):
        """
        Initialize POS tagger.
        :param lang: "en" for English, "fa" for Persian
        """
        self.lang = lang

        if self.lang == "en":
            # correct resource names
            nltk.download('averaged_perceptron_tagger', quiet=True)
            nltk.download('punkt', quiet=True)

        elif self.lang == "fa":
            # Prefer using an existing Hazm tagger (my_tagger) if available in the notebook.
            # Otherwise try to initialize a stanza pipeline (do NOT auto-download models here).
            try:
                from hazm import POSTagger as HazmPOSTagger
                if 'my_tagger' in globals() and isinstance(globals()['my_tagger'], HazmPOSTagger):
                    # use the Hazm tagger already loaded in the notebook
                    self.hazm = globals()['my_tagger']
                    self.backend = 'hazm'
                    return
            except Exception:
                # Hazm not available or my_tagger not present; fall back to stanza attempt below
                pass

            # Try to initialize stanza Pipeline. If stanza models are missing, do NOT attempt automatic download here
            # (automatic download in some environments can trigger model-loading code paths that raise unexpected errors).
            try:
                self.nlp = stanza.Pipeline(lang="fa", processors="tokenize,pos", verbose=False)
                self.backend = 'stanza'
            except Exception as e:
                raise RuntimeError(
                    "Failed to initialize a Persian tagger. "
                    "If you want to use Hazm, make sure 'my_tagger' (hazm.pos_tagger.POSTagger) is available. "
                    "To use stanza, please run stanza.download('fa') in a separate cell and ensure stanza models are installed. "
                    f"Original error: {e}"
                )

        else:
            raise ValueError("Supported languages are 'en' and 'fa'.")

    def tag(self, text: str):
        """
        Return POS tags for given text.
        :param text: input string
        :return: list of (word, pos_tag)
        """
        if self.lang == "en":
            tokens = nltk.word_tokenize(text)
            return nltk.pos_tag(tokens)

        elif self.lang == "fa":
            # If using Hazm tagger
            if getattr(self, "backend", None) == 'hazm':
                from hazm import word_tokenize as hazm_word_tokenize
                tokens = hazm_word_tokenize(text)
                return self.hazm.tag(tokens)

            # If using stanza pipeline
            elif getattr(self, "backend", None) == 'stanza':
                doc = self.nlp(text)
                result = []
                for sent in doc.sentences:
                    for word in sent.words:
                        result.append((word.text, word.upos))
                return result

            else:
                raise RuntimeError("Persian tagger backend is not initialized.")

# Instantiate English tagger (this will download the correct NLTK resources if needed)
en_tagger = POSTagger("en")
print(en_tagger.tag("This is a simple English sentence."))

# For Persian, if you already have 'my_tagger' (Hazm POSTagger) in the notebook, POSTagger("fa") will use it.
# Otherwise, ensure you run stanza.download('fa') in a separate cell before creating the stanza-based tagger.
fa_tagger = POSTagger("fa")
print(fa_tagger.tag("من امروز به دانشگاه رفتم."))

[('This', 'DT'), ('is', 'VBZ'), ('a', 'DT'), ('simple', 'JJ'), ('English', 'JJ'), ('sentence', 'NN'), ('.', '.')]
[('من', 'PRON'), ('امروز', 'ADV'), ('به', 'ADP'), ('دانشگاه', 'NOUN'), ('رفتم', 'VERB'), ('.', 'PUNCT')]


In [37]:
print(en_tagger.tag("You Know nothing, John Snow!"))

[('You', 'PRP'), ('Know', 'VBP'), ('nothing', 'NN'), (',', ','), ('John', 'NNP'), ('Snow', 'NNP'), ('!', '.')]


In [38]:
# Download the model file from Google Drive
!gdown https://drive.google.com/uc?id=1Q3JK4NVUC2t5QT63aDiVrCRBV225E_B3 -O pos_tagger.model

Downloading...
From: https://drive.google.com/uc?id=1Q3JK4NVUC2t5QT63aDiVrCRBV225E_B3
To: d:\josef\OneDrive\Desktop\Projects 2025\Rahnama\Git\text-preprocessing-nlp\pos_tagger.model

  0%|          | 0.00/19.2M [00:00<?, ?B/s]
  3%|▎         | 524k/19.2M [00:00<00:33, 563kB/s]
  5%|▌         | 1.05M/19.2M [00:01<00:17, 1.03MB/s]
  8%|▊         | 1.57M/19.2M [00:01<00:12, 1.39MB/s]
 11%|█         | 2.10M/19.2M [00:01<00:10, 1.65MB/s]
 14%|█▎        | 2.62M/19.2M [00:02<00:11, 1.48MB/s]
 19%|█▉        | 3.67M/19.2M [00:02<00:06, 2.31MB/s]
 25%|██▍       | 4.72M/19.2M [00:02<00:04, 3.05MB/s]
 30%|██▉       | 5.77M/19.2M [00:02<00:03, 3.56MB/s]
 35%|███▌      | 6.82M/19.2M [00:02<00:03, 3.83MB/s]
 38%|███▊      | 7.34M/19.2M [00:03<00:03, 3.61MB/s]
 41%|████      | 7.86M/19.2M [00:03<00:03, 3.32MB/s]
 44%|████▎     | 8.39M/19.2M [00:03<00:03, 3.15MB/s]
 46%|████▋     | 8.91M/19.2M [00:03<00:03, 3.28MB/s]
 52%|█████▏    | 9.96M/19.2M [00:03<00:02, 3.48MB/s]
 54%|█████▍    | 10.5M/19.2M [00:

In [41]:
from hazm import POSTagger, word_tokenize # Import word_tokenize

# Use Hazm's POSTagger with the downloaded model
my_tagger = POSTagger(model='pos_tagger.model')

# Example Persian text
persian_text = "من امروز به دانشگاه رفتم."

# Tokenize the text using Hazm's word_tokenize
tokens = word_tokenize(persian_text)

# Tag the tokens using the Hazm tagger with the loaded model
pos_tags = my_tagger.tag(tokens) # Hazm tagger expects tokens

# Print the results
print(pos_tags)

[('من', 'PRON'), ('امروز', 'ADV'), ('به', 'ADP'), ('دانشگاه', 'NOUN'), ('رفتم', 'VERB'), ('.', 'PUNCT')]


## Data loading and initial cleaning

### Subtask:
Load your cleaned data. This might involve reading text from files, databases, or other sources. Perform any final cleaning steps if necessary (e.g., handling special characters, normalization).


In [6]:
!pip install datasets

In [7]:
from datasets import load_dataset

ds = load_dataset("shenasa/English-Persian-Parallel-Dataset")

README.md: 0.00B [00:00, ?B/s]

c:\Users\josef\.conda\envs\rc\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\josef\.cache\huggingface\hub\datasets--shenasa--English-Persian-Parallel-Dataset. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to

dataset.tsv:   0%|          | 0.00/872M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3960172 [00:00<?, ? examples/s]

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [29]:
ds

DatasetDict({
    train: Dataset({
        features: ['flash fire .', 'فلاش آتش .'],
        num_rows: 3960172
    })
})

In [9]:
ds.save_to_disk(".ds_dataset")
print("Dataset saved to the folder 'ds_dataset'.")

Saving the dataset (0/2 shards):   0%|          | 0/3960172 [00:00<?, ? examples/s]

Dataset saved to the folder 'ds_dataset'.


In [10]:
import pandas as pd

In [48]:
ds['train']

Dataset({
    features: ['flash fire .', 'فلاش آتش .'],
    num_rows: 3960172
})

In [12]:

p = ds.data['train'].to_pandas()

In [13]:
p.rename(columns={'flash fire .':'en','فلاش آتش .':'fa' },inplace=True)

In [40]:
p

,en,fa
0,superheats the air . burns the lungs like rice...,هوا را فوق العاده گرم می کند . ریه ها را مثل ک...
1,"hey , guys . down here . down here .",سلام بچه ها . این پایین . این پایین .
2,what do you got down this corridor is the bow ...,چه چیزی در این راهرو پایین آمده است ، درست است .
3,theres an access hatch right there that puts u...,یک دریچه دسترسی درست در آنجا وجود دارد که ما ر...
4,we get into the propeller tubes and the only t...,وارد لوله های پروانه می شویم و تنها چیزی که بی...
...,...,...
3960167,The devices are connected in a daisy chain tal...,دستگاه ها به صورت زنجیره ای به هم متصل شده اند...
3960168,Learn more about our SC Net,درباره SC Net ما بیشتر بیاموزید
3960169,Premium quality third party products.,محصولات شخص ثالث با کیفیت برتر .
3960170,Compact. Lightweight. Protected.,فشرده . سبک وزن . محافظت شده است .


In [55]:
import pandas as pd
import json

def pos_tag_dataframe_to_csv(df, fa_tagger, en_tagger, output_file, batch_size=50):
    """
    POS tag Persian-English sentences in a DataFrame and save results to CSV in batches.
    Skips any rows where data is invalid or tagging fails for a row.

    :param df: DataFrame with columns ['fa', 'en']
    :param fa_tagger: Persian POS tagger
    :param en_tagger: English POS tagger
    :param output_file: path to CSV file
    :param batch_size: number of rows per batch
    """
    first_write = True
    total = 0

    for start in range(0, len(df), batch_size):
        end = min(start + batch_size, len(df))
        df_chunk = df.iloc[start:end]

        chunk_results = []
        for local_idx, row in df_chunk.iterrows():
            global_idx = start + local_idx

            # Validate columns exist and are strings
            try:
                fa_sent = row['fa']
                en_sent = row['en']
            except Exception:
                print(f"Skipping row {global_idx}: missing 'fa' or 'en' column")
                continue

            if not isinstance(fa_sent, str) or not isinstance(en_sent, str):
                print(f"Skipping row {global_idx}: non-string data (fa: {type(fa_sent)}, en: {type(en_sent)})")
                continue

            if not fa_sent.strip() or not en_sent.strip():
                print(f"Skipping row {global_idx}: empty sentence")
                continue

            # Attempt tagging; skip row if tagging fails
            try:
                fa_tags = fa_tagger.tag(fa_sent)
            except Exception as e:
                print(f"Skipping row {global_idx}: Persian tagging error: {e}")
                continue

            try:
                en_tags = en_tagger.tag(en_sent)
            except Exception as e:
                print(f"Skipping row {global_idx}: English tagging error: {e}")
                continue

            # Ensure tags are JSON-serializable (convert tuples to lists)
            try:
                fa_serializable = [list(t) if isinstance(t, tuple) else t for t in fa_tags]
            except Exception:
                fa_serializable = fa_tags

            try:
                en_serializable = [list(t) if isinstance(t, tuple) else t for t in en_tags]
            except Exception:
                en_serializable = en_tags

            chunk_results.append({
                "fa_sentence": fa_sent,
                "fa_tags": json.dumps(fa_serializable, ensure_ascii=False),
                "en_sentence": en_sent,
                "en_tags": json.dumps(en_serializable, ensure_ascii=False)
            })

        # Only write if we have successful results in this chunk
        if chunk_results:
            tagged_chunk = pd.DataFrame(chunk_results)
            tagged_chunk.to_csv(
                output_file,
                mode="a",
                index=False,
                header=first_write,
                encoding="utf-8"
            )
            first_write = False
        else:
            print(f"No valid rows in batch {start}-{end}, nothing written.")

        total += len(df_chunk)
        print(f"Processed up to row {end}/{len(df)} (attempted {len(df_chunk)} rows; wrote {len(chunk_results)} valid rows)")

    print(f"✅ Finished tagging. Results saved in {output_file}")


In [ ]:
pos_tag_dataframe_to_csv(
    p,
    fa_tagger,
    en_tagger,
    output_file="tagged_dataset.csv",
    # output_file="/content/drive/MyDrive/tagged_dataset.csv",
    batch_size=10_000
)

Processed up to row 10000/3960172 (attempted 10000 rows; wrote 10000 valid rows)
Processed up to row 20000/3960172 (attempted 10000 rows; wrote 10000 valid rows)
Processed up to row 20000/3960172 (attempted 10000 rows; wrote 10000 valid rows)
Processed up to row 30000/3960172 (attempted 10000 rows; wrote 10000 valid rows)
Processed up to row 30000/3960172 (attempted 10000 rows; wrote 10000 valid rows)
Processed up to row 40000/3960172 (attempted 10000 rows; wrote 10000 valid rows)
Processed up to row 40000/3960172 (attempted 10000 rows; wrote 10000 valid rows)
Processed up to row 50000/3960172 (attempted 10000 rows; wrote 10000 valid rows)
Processed up to row 50000/3960172 (attempted 10000 rows; wrote 10000 valid rows)
Processed up to row 60000/3960172 (attempted 10000 rows; wrote 10000 valid rows)
Processed up to row 60000/3960172 (attempted 10000 rows; wrote 10000 valid rows)
Processed up to row 70000/3960172 (attempted 10000 rows; wrote 10000 valid rows)
Processed up to row 70000/39

In [53]:
for start in range(0, len(ds), 10000):
    end = min(start + 10000, len(ds))
    print(start,end)

0 1
